# Taller 1 — Huella hídrica agrícola y propiedades físicas del suelo

**Asignatura:** TAG2027 — Relación Suelo Agua Planta | UDLA  
**Unidades:** 1 (El agua en la producción agrícola) + inicio Unidad 2 (El agua en el suelo)  
**Duración:** 3 horas  
**Ejercicio vinculado:** Ejercicio 1

---

## Objetivos

Al finalizar este taller serás capaz de:
1. Calcular la huella hídrica de productos agrícolas chilenos
2. Calcular densidad aparente, densidad real y porosidad total del suelo
3. Determinar el contenido gravimétrico de agua en muestras de suelo
4. Usar Google Colab como entorno de cálculo para los talleres del curso

---

## Instrucciones

1. Ejecuta cada celda en orden (Shift + Enter)
2. Completa las celdas marcadas con `# TODO`
3. Al finalizar, descarga el notebook: Archivo → Descargar → .ipynb

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frzambra/RSPA-TAG-UDLA/blob/main/labs/taller-01/taller-01.ipynb)

In [ ]:
# ========================================
# Celda de setup — Ejecutar primero
# ========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configuración de gráficos
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

# Función auxiliar para imprimir resultados con unidades
def resultado(valor, unidad, nombre):
    print(f'{nombre}: {valor:.3f} {unidad}')

print('✓ Setup completo — numpy, pandas y matplotlib listos.')

---
## Parte 1: Huella hídrica agrícola (45 min)

La **huella hídrica** de un producto agrícola es el volumen total de agua dulce utilizado para producirlo, desde el cultivo hasta la cosecha.

$$HF_{total} = HF_{azul} + HF_{verde} + HF_{gris}$$

| Componente | Descripción |
|-----------|-------------|
| Huella azul | Agua superficial y subterránea consumida (riego) |
| Huella verde | Agua de lluvia almacenada en el suelo |
| Huella gris | Agua necesaria para diluir contaminantes |

### Datos de referencia

Usamos datos de Water Footprint Network para cultivos representativos chilenos:

In [ ]:
# Datos de huella hídrica para productos agrícolas chilenos
# Fuente: Water Footprint Network (waterfootprint.org) — valores promedio global
# Unidades: litros de agua por kg de producto

huella = pd.DataFrame({
    'Producto': ['Palta', 'Uva de mesa', 'Maíz', 'Trigo', 'Manzana', 'Tomate', 'Lechuga'],
    'HF_azul': [850, 210, 200, 350, 130, 60, 20],    # L/kg — agua de riego
    'HF_verde': [1150, 390, 700, 950, 360, 120, 220], # L/kg — agua de lluvia
    'HF_gris': [80, 30, 50, 70, 20, 15, 10]           # L/kg — dilución contaminantes
})

huella['HF_total'] = huella['HF_azul'] + huella['HF_verde'] + huella['HF_gris']
huella

### Ejercicio 1.1: Comparación de huella hídrica

Visualiza la huella hídrica total por producto.

In [ ]:
# SOLUCIÓN — Ejercicio 1.1: Gráfico de barras horizontales de HF_total

huella_sorted = huella.sort_values('HF_total')

fig, ax = plt.subplots()
barras = ax.barh(huella_sorted['Producto'], huella_sorted['HF_total'], color='steelblue')
ax.set_xlabel('Huella hídrica total (L/kg)')
ax.set_title('Huella hídrica total por producto agrícola')

for barra, valor in zip(barras, huella_sorted['HF_total']):
    ax.text(barra.get_width() + 20, barra.get_y() + barra.get_height() / 2,
            f'{valor:,.0f}', va='center')

plt.tight_layout()
plt.show()


### Ejercicio 1.2: Agua virtual exportada

Chile exporta aproximadamente **180,000 toneladas de palta** al año.

Calcula el agua virtual exportada en paltas (en m³ y en equivalente a casas).

Dato: Una casa en Chile consume aproximadamente 180 m³ de agua al año.

In [ ]:
# Datos del problema
exportacion_palta_ton = 180_000  # toneladas/año
hf_palta_total = huella[huella['Producto'] == 'Palta']['HF_total'].values[0]  # L/kg
consumo_casa_m3 = 180  # m³/año por casa

# SOLUCIÓN — Cálculo de agua virtual exportada
kg_palta = exportacion_palta_ton * 1000          # 1 tonelada = 1000 kg
litros_virtuales = kg_palta * hf_palta_total     # agua total en litros
m3_virtuales = litros_virtuales / 1000           # 1 m³ = 1000 L
casas_equivalentes = m3_virtuales / consumo_casa_m3

print(f'Agua virtual exportada en palta: {m3_virtuales:,.0f} m³/año')
print(f'Equivalente al consumo de {casas_equivalentes:,.0f} casas')


### Ejercicio 1.3: Discusión

Responde en una celda de texto:
1. ¿Por qué la palta tiene una huella hídrica tan alta?
2. ¿Qué implicancias tiene para Chile exportar tanta agua virtual en forma de fruta?
3. ¿Cómo se podría reducir la huella hídrica azul de un cultivo?

### Respuestas sugeridas (Ejercicio 1.3)

**1. ¿Por qué la palta tiene una huella hídrica tan alta?**
La palta requiere del orden de 2.000 L/kg porque es un fruto de desarrollo lento y alto contenido graso (mucha biomasa por unidad de peso). Además se cultiva en zonas de clima mediterráneo/seco con alta demanda evaporativa, donde el agua de lluvia no basta y depende casi por completo del riego (huella azul elevada).

**2. ¿Qué implicancias tiene para Chile exportar tanta agua virtual en fruta?**
Exportar palta equivale a exportar "agua virtual": el agua que se consumió para producirla. En regiones productoras como Valparaíso esto presiona acuíferos y ríos, genera estrés hídrico y conflictos por el agua. Es una decisión de asignación de un recurso escaso: se destina agua local a abastecer mercados externos.

**3. ¿Cómo se podría reducir la huella hídrica azul de un cultivo?**
- Riego tecnificado (goteo en lugar de surco o tendido).
- Programar el riego según la demanda real del cultivo (balance hídrico, sensores de humedad).
- Mejorar la retención de agua del suelo (materia orgánica, cobertura, mulching).
- Usar variedades más eficientes en el uso del agua y captar aguas lluvias.


---
## Parte 2: Densidad aparente y porosidad (60 min)

### Conceptos

**Densidad aparente (Da):** Masa de suelo seco por unidad de volumen total (incluyendo poros).

$$D_a = \frac{M_{seco}}{V_{total}} \quad [\text{g/cm}^3]$$

**Densidad real (Dr):** Masa de suelo seco por unidad de volumen de partículas sólidas (sin poros).

$$D_r = \frac{M_{seco}}{V_{solidos}} \quad [\text{g/cm}^3]$$

**Porosidad total (P):** Fracción del volumen del suelo ocupada por poros.

$$P = \left(1 - \frac{D_a}{D_r}\right) \times 100 \quad [\%]$$

### Valores típicos

| Tipo de suelo | Da (g/cm³) | Dr (g/cm³) | Porosidad (%) |
|--------------|------------|------------|---------------|
| Arenoso | 1.5 - 1.8 | 2.65 | 30 - 43 |
| Franco | 1.2 - 1.5 | 2.65 | 43 - 55 |
| Arcilloso | 0.9 - 1.2 | 2.65 | 55 - 66 |
| Orgánico | 0.3 - 0.8 | 1.5 - 2.0 | 60 - 85 |

In [ ]:
# Datos de muestras de suelo para el ejercicio
# Muestra obtenida con cilindro metálico de volumen conocido

muestras = pd.DataFrame({
    'Muestra': ['Suelo A (arenoso)', 'Suelo B (franco)', 'Suelo C (arcilloso)', 'Suelo D (franco-arenoso)'],
    'Vol_cilindro_cm3': [100, 100, 100, 100],       # cm³
    'Masa_suelo_humedo_g': [185.0, 172.0, 155.0, 178.0],  # g
    'Masa_suelo_seco_g': [165.0, 142.0, 118.0, 153.0],     # g (secado a 105°C por 24h)
    'Dr_g_cm3': [2.65, 2.65, 2.65, 2.65]                  # g/cm³ (densidad real ≈ cuarzo)
})
muestras

In [ ]:
# SOLUCIÓN — Densidad aparente: Da = Masa_seco / Volumen_cilindro
muestras['Da_g_cm3'] = muestras['Masa_suelo_seco_g'] / muestras['Vol_cilindro_cm3']

muestras[['Muestra', 'Da_g_cm3']]


In [ ]:
# Celda de validación — ejecuta para verificar tus resultados
# Da esperados: ~1.65, ~1.42, ~1.18, ~1.53 g/cm³
if 'Da_g_cm3' in muestras.columns:
    da_ok = np.allclose(muestras['Da_g_cm3'], [1.65, 1.42, 1.18, 1.53], atol=0.02)
    print('✓ Densidad aparente correcta' if da_ok else '⚠ Revisa los cálculos de Da')
    print(f'\nTus valores:\n{muestras[["Muestra", "Da_g_cm3"]].to_string(index=False)}')
else:
    print('⚠ Primero calcula Da_g_cm3')

In [ ]:
# SOLUCIÓN — Porosidad total: P = (1 - Da/Dr) * 100
muestras['Porosidad_pct'] = (1 - muestras['Da_g_cm3'] / muestras['Dr_g_cm3']) * 100

muestras[['Muestra', 'Da_g_cm3', 'Porosidad_pct']]


In [ ]:
# Celda de validación — porosidad esperada
if 'Porosidad_pct' in muestras.columns:
    p_ok = np.allclose(muestras['Porosidad_pct'], [37.74, 46.42, 55.47, 42.26], atol=0.5)
    print('✓ Porosidad correcta' if p_ok else '⚠ Revisa los cálculos de P')
else:
    print('⚠ Primero calcula Porosidad_pct')

### Pregunta de interpretación

¿Qué relación observas entre la textura del suelo y su densidad aparente y porosidad? Responde en una celda de texto.

### Respuesta sugerida (relación textura ↔ Da y porosidad)

| Suelo | Da (g/cm³) | Porosidad (%) |
|-------|-----------|---------------|
| A (arenoso) | 1.65 | 37.7 |
| D (franco-arenoso) | 1.53 | 42.3 |
| B (franco) | 1.42 | 46.4 |
| C (arcilloso) | 1.18 | 55.5 |

A medida que el suelo es más **arcilloso** (partículas más finas), la densidad aparente **disminuye** y la porosidad total **aumenta**. Las partículas finas forman agregados y microporos que mantienen un mayor volumen de poros; las arenas, con partículas gruesas y poco estructuradas, se compactan más (mayor Da, menor porosidad). Existe por tanto una relación **inversa** entre Da y porosidad, descrita por P = (1 − Da/Dr) × 100.


---
## Parte 3: Contenido gravimétrico de agua (45 min)

### Concepto

El **contenido gravimétrico de agua** ($\theta_g$) es la masa de agua por unidad de masa de suelo seco:

$$\theta_g = \frac{M_{humedo} - M_{seco}}{M_{seco}} \quad [\text{g/g}]$$

Se determina pesando una muestra antes y después de secarla en estufa a 105°C durante 24 horas.

### Datos del ejercicio

Se tomaron 5 muestras de suelo en un campo de maíz. Cada muestra se pesó inmediatamente (húmeda) y después de secado en estufa:

In [ ]:
# Datos de contenido de humedad — muestras de terreno
humedad = pd.DataFrame({
    'Punto': [1, 2, 3, 4, 5],
    'Masa_humedo_g': [87.5, 92.3, 78.1, 95.0, 83.2],
    'Masa_seco_g': [72.0, 74.1, 65.5, 76.3, 69.8]
})
humedad['Masa_agua_g'] = humedad['Masa_humedo_g'] - humedad['Masa_seco_g']
humedad

In [ ]:
# SOLUCIÓN — Contenido gravimétrico: θg = Masa_agua / Masa_seco
humedad['Theta_g_gg'] = humedad['Masa_agua_g'] / humedad['Masa_seco_g']

humedad[['Punto', 'Theta_g_gg']]


In [ ]:
# Celda de validación
if 'Theta_g_gg' in humedad.columns:
    tg_ok = np.allclose(humedad['Theta_g_gg'], [0.215, 0.246, 0.192, 0.245, 0.192], atol=0.01)
    print('✓ θg correcto' if tg_ok else '⚠ Revisa los cálculos de θg')
else:
    print('⚠ Primero calcula Theta_g_gg')

In [ ]:
# Visualización de θg por punto de muestreo
# SOLUCIÓN — gráfico de barras con etiquetas de valor

if 'Theta_g_gg' in humedad.columns:
    fig, ax = plt.subplots()
    barras = ax.bar(humedad['Punto'], humedad['Theta_g_gg'], color='seagreen')
    ax.set_xlabel('Punto de muestreo')
    ax.set_ylabel('Contenido gravimétrico θg (g/g)')
    ax.set_title('Contenido gravimétrico de agua por punto de muestreo')

    for barra, valor in zip(barras, humedad['Theta_g_gg']):
        ax.text(barra.get_x() + barra.get_width() / 2, barra.get_height() + 0.003,
                f'{valor:.3f}', ha='center', va='bottom')

    ax.set_ylim(0, humedad['Theta_g_gg'].max() * 1.15)
    plt.tight_layout()
    plt.show()
else:
    print('⚠ Primero calcula Theta_g_gg')


### Variabilidad espacial de la humedad

Calcula la media, desviación estándar y coeficiente de variación (CV) de θg:

In [ ]:
# SOLUCIÓN — Estadística descriptiva de θg
# media, desviación estándar y coeficiente de variación

if 'Theta_g_gg' in humedad.columns:
    media_tg = humedad['Theta_g_gg'].mean()
    desv_tg = humedad['Theta_g_gg'].std()
    cv_tg = (desv_tg / media_tg) * 100

    print(f'Media de θg: {media_tg:.3f} g/g')
    print(f'Desviación estándar: {desv_tg:.3f} g/g')
    print(f'Coeficiente de variación: {cv_tg:.1f}%')

    # Interpretación:
    # CV < 15% → variabilidad baja
    # CV 15-35% → variabilidad moderada
    # CV > 35% → variabilidad alta
    if cv_tg < 15:
        print('\nInterpretación: variabilidad baja')
    elif cv_tg <= 35:
        print('\nInterpretación: variabilidad moderada')
    else:
        print('\nInterpretación: variabilidad alta')
else:
    print('⚠ Primero calcula Theta_g_gg')


---
## Parte 4: Setup computacional (30 min)

### Verifica tu entorno

Ejecuta las siguientes celdas para asegurarte de que tienes todo listo para los próximos talleres.

In [ ]:
# Verificación de versiones de librerías
import sys
print(f'Python: {sys.version}')
print(f'numpy:  {np.__version__}')
print(f'pandas: {pd.__version__}')

import matplotlib
print(f'matplotlib: {matplotlib.__version__}')

# Verificar que scipy está disponible (lo usaremos en talleres posteriores)
try:
    import scipy
    print(f'scipy:  {scipy.__version__}')
except ImportError:
    print('⚠ scipy no está instalado. En Taller 4 lo necesitaremos.')
    print('  Ejecuta: !pip install scipy')

In [ ]:
# Prueba: cargar datos desde una URL raw de GitHub
# Esto es lo que haremos en talleres posteriores
try:
    url_ejemplo = 'https://raw.githubusercontent.com/datasets/co2-fossil-global/master/global.csv'
    df_test = pd.read_csv(url_ejemplo)
    print('✓ Carga de datos desde URL funcional')
    print(f'  Filas cargadas: {len(df_test)}')
except Exception as e:
    print(f'No se pudo cargar (normal si no hay internet): {e}')

---
## Entregable del Taller 1

Al finalizar, entrega este notebook con todas las celdas completas.

**Debes incluir:**
1. Gráfico de huella hídrica por producto (Ejercicio 1.1)
2. Cálculo de agua virtual exportada en palta (Ejercicio 1.2) ✅
3. Discusión sobre huella hídrica (Ejercicio 1.3)
4. Cálculo de Da y porosidad para 4 suelos (Parte 2)
5. Interpretación de la relación textura-porosidad
6. Cálculo de θg y estadística descriptiva (Parte 3)
7. Análisis de variabilidad espacial

**Formato de entrega:** Descarga el notebook (.ipynb) y súbelo a la plataforma del curso.

---
*Taller 1 — TAG2027 Relación Suelo Agua Planta | UDLA*